In [1]:
print("AutoModelForVision")

AutoModelForVision


In [5]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import torch

# ── AUTOMODEL FOR IMAGE CLASSIFICATION (google/vit-base-patch16-224) ─────────
# Task         : Image classification — 1000 ImageNet classes
# Architecture : ViT (Vision Transformer) — treats image as a sequence of patches
#                Image split into 16×16 pixel patches → each patch = one "token"
#                224×224 image → (224/16)² = 196 patches + 1 [CLS] token = 197 tokens
#                12 transformer layers, hidden size 768
# Head         : Linear layer on top of [CLS] token → 1000 output logits
# Pre-trained  : ImageNet-21k → fine-tuned on ImageNet-1k (ILSVRC)
# Output       : raw logits → softmax → probabilities → argmax → label
# ─────────────────────────────────────────────────────────────────────────────


# ── STEP 1 : Load processor ───────────────────────────────────────────────────
# AutoImageProcessor — image equivalent of AutoTokenizer
# Handles all preprocessing that the model expects:
#   resize     → 224×224  (ViT's fixed input resolution)
#   normalize  → pixel values scaled using ImageNet mean/std per channel
#                mean = [0.485, 0.456, 0.406]  std = [0.229, 0.224, 0.225]
#   to tensor  → HWC PIL image → CHW PyTorch float tensor
# Must use the same processor config used during pre-training — mismatched
# normalization silently corrupts inputs and tanks accuracy
processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")

print("─" * 60)
print("STEP 1 — Processor loaded")
print(f"  Processor class  : {type(processor).__name__}")
print(f"  Image size       : {processor.size}")
print(f"  Rescale factor   : {processor.rescale_factor}")


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForImageClassification adds a classification head on top of ViT
# Downloads config.json → architecture details (patch_size, num_labels, id2label)
#          pytorch_model.bin / model.safetensors → pre-trained weights
# model.eval() → disables dropout for deterministic inference
#   training   : dropout randomly zeros activations → acts as regularisation
#   inference  : must be OFF — full, stable activations required
model = AutoModelForImageClassification.from_pretrained("google/vit-base-patch16-224")
model.eval()   # IMPORTANT — always call before inference

print("\nSTEP 2 — Model loaded")
print(f"  Model class      : {type(model).__name__}")
print(f"  Num labels       : {model.config.num_labels}")
#   1000 — full ImageNet-1k label set
print(f"  Sample labels    : { {k: model.config.id2label[k] for k in range(3)} }")
#   {0: 'tench', 1: 'goldfish', 2: 'great white shark'}


# ── STEP 3 : Load and preprocess image ───────────────────────────────────────
# PIL Image.open → loads raw image from disk in HWC format (Height × Width × Channels)
# processor() wraps three operations in one call:
#   1. Resize       : PIL resize to 224×224
#   2. Rescale      : uint8 [0, 255] → float32 [0.0, 1.0]  (rescale_factor = 1/255)
#   3. Normalize    : (pixel - mean) / std  per channel (ImageNet statistics)
# return_tensors="pt" → output is a dict of PyTorch tensors
#   pixel_values shape : [batch_size, channels, height, width] = [1, 3, 224, 224]
image = Image.open("kingfisher.png")
inputs = processor(images=image, return_tensors="pt")

print("\nSTEP 3 — Image loaded and preprocessed")
print(f"  Original size    : {image.size}")        # (W, H) e.g. (640, 480)
print(f"  pixel_values     : {inputs['pixel_values'].shape}")
#   torch.Size([1, 3, 224, 224])
print(f"  Dtype            : {inputs['pixel_values'].dtype}")
#   torch.float32


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() → disables gradient tracking for all ops inside the block
#   training   : gradients tracked so .backward() can update weights
#   inference  : .backward() never called — tracking wastes memory and time
#   no_grad cuts memory ~50% and speeds up the forward pass noticeably
#
# Inside the model — what ViT actually does:
#   1. Patch embed  : 224×224 image → 196 flat patch vectors of size 768
#   2. Prepend [CLS]: [CLS] token concat → sequence length = 197
#   3. Add pos embed: learnable position embeddings added to all 197 tokens
#   4. Transformer  : 12 layers of (multi-head self-attention → MLP) with LayerNorm
#   5. [CLS] output : take hidden state of position 0 → shape [1, 768]
#   6. Classifier   : Linear(768 → 1000) → raw logits
with torch.no_grad():
    outputs = model(**inputs)

# outputs is an ImageClassifierOutput dataclass
# .logits → raw scores before any activation, shape [batch_size, num_labels]
#   higher score = model assigns more confidence to that class
#   NOT probabilities yet — can be any real number
logits = outputs.logits   # shape: [1, 1000]

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type      : {type(outputs).__name__}")
print(f"  Logits shape     : {logits.shape}")
#   torch.Size([1, 1000])


# ── STEP 5 : Post-process — top prediction ───────────────────────────────────
# argmax across 1000 classes → index of highest logit → predicted class id
# id2label maps integer id → human-readable ImageNet class string
pred_id = logits.argmax(-1).item()   # .item() → plain Python int
label   = model.config.id2label[pred_id]

print("\nSTEP 5 — Top prediction")
print(f"  Predicted id     : {pred_id}")
print(f"  Predicted label  : {label}")
#   "tabby, tabby cat"


# ── STEP 6 : Post-process — top-5 predictions ────────────────────────────────
# logits[0] → squeeze batch dim → shape [1000]
# torch.softmax → converts raw scores to probabilities summing to 1.0
#   softmax(x_i) = exp(x_i) / Σ exp(x_j)
#   preserves relative order — highest logit still wins
# .topk(5) → returns (values, indices) of the 5 largest probabilities
#   values  : top-5 probability scores
#   indices : corresponding class ids
probs = torch.softmax(logits[0], dim=0)
top5  = probs.topk(5)

print("\nSTEP 6 — Top-5 predictions")
print("─" * 60)
for prob, idx in zip(top5.values, top5.indices):
    class_label = model.config.id2label[idx.item()]
    print(f"  {class_label:<40s} — {prob.item():.4f}")
#   tabby, tabby cat                         — 0.6213
#   tiger cat                                — 0.1847
#   Egyptian cat                             — 0.0934
#   lynx, catamount                          — 0.0312
#   Persian cat                              — 0.0271
print("─" * 60)


# ── VIT PATCH BREAKDOWN — DRY RUN ────────────────────────────────────────────
# Input image  : "cat.jpg" — assumed 640×480 RGB
#
# Stage 1 — Resize + Normalize (inside processor)
# 640×480 → resize → 224×224
# each pixel : (value/255 - mean) / std  per R, G, B channel
# output     : float32 tensor [1, 3, 224, 224]
#
# Stage 2 — Patch embedding (inside model)
# 224×224 image split into 14×14 grid of 16×16 patches = 196 patches
# each patch : flatten 16×16×3 = 768 values → Linear(768 → 768)
# prepend [CLS] token → sequence of 197 vectors, each dim 768
#
# Stage 3 — 12 Transformer layers
# each layer applies to all 197 tokens:
#   Multi-Head Self-Attention (12 heads, each head dim = 64)
#   Add & LayerNorm
#   MLP : Linear(768 → 3072) → GELU → Linear(3072 → 768)
#   Add & LayerNorm
#
# Stage 4 — Classification head
# take [CLS] hidden state → shape [1, 768]
# Linear(768 → 1000) → logits shape [1, 1000]
# argmax → class id → id2label → "tabby, tabby cat"

────────────────────────────────────────────────────────────
STEP 1 — Processor loaded
  Processor class  : ViTImageProcessor
  Image size       : SizeDict(height=224, width=224, longest_edge=None, shortest_edge=None, max_height=None, max_width=None)
  Rescale factor   : 0.00392156862745098


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]


STEP 2 — Model loaded
  Model class      : ViTForImageClassification
  Num labels       : 1000
  Sample labels    : {0: 'tench, Tinca tinca', 1: 'goldfish, Carassius auratus', 2: 'great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias'}

STEP 3 — Image loaded and preprocessed
  Original size    : (740, 1109)
  pixel_values     : torch.Size([1, 3, 224, 224])
  Dtype            : torch.float32

STEP 4 — Forward pass complete
  Output type      : ImageClassifierOutput
  Logits shape     : torch.Size([1, 1000])

STEP 5 — Top prediction
  Predicted id     : 88
  Predicted label  : macaw

STEP 6 — Top-5 predictions
────────────────────────────────────────────────────────────
  macaw                                    — 0.9598
  lorikeet                                 — 0.0363
  African grey, African gray, Psittacus erithacus — 0.0010
  sulphur-crested cockatoo, Kakatoe galerita, Cacatua galerita — 0.0006
  toucan                                   — 0.0002
─────

In [6]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from PIL import Image
import torch

# ── AUTOMODEL FOR OBJECT DETECTION (facebook/detr-resnet-50) ─────────────────
# Task         : Object detection — locate + classify multiple objects in one image
# Architecture : DETR (DEtection TRansformer) — end-to-end detection, no NMS needed
#                Backbone  : ResNet-50 — extracts spatial feature map from image
#                Encoder   : Transformer encoder — models global context over features
#                Decoder   : Transformer decoder — 100 learned object queries attend
#                            to encoder output → each query predicts one object
# Head         : Two parallel linear layers on each query output:
#                  class head : Linear(256 → num_labels + 1)  — +1 for "no object"
#                  bbox head  : MLP(256 → 4)  — predicts (cx, cy, w, h) normalised
# Trained on   : COCO 2017 — 80 object categories
# Key insight  : Object queries are learnable slots — model learns which slot
#                specialises in which region/size over training
# Output       : logits + pred_boxes → post-processed into scores, labels, boxes
# ─────────────────────────────────────────────────────────────────────────────


# ── STEP 1 : Load processor ───────────────────────────────────────────────────
# AutoImageProcessor for DETR wraps:
#   resize      → shortest edge scaled to 800px, longest edge capped at 1333px
#                 aspect ratio preserved (unlike ViT's fixed 224×224 square crop)
#   normalize   → ImageNet mean/std per channel (same as ViT)
#                 mean = [0.485, 0.456, 0.406]  std = [0.229, 0.224, 0.225]
#   to tensor   → PIL HWC → PyTorch float CHW tensor
# Also handles padding inside a batch so all images reach the same spatial size
processor = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")

print("─" * 60)
print("STEP 1 — Processor loaded")
print(f"  Processor class  : {type(processor).__name__}")
print(f"  Resize config    : {processor.size}")
#   {'shortest_edge': 800}  — variable resolution, not a fixed square


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForObjectDetection adds the detection head on top of DETR backbone
# Downloads config.json  → architecture details (num_queries=100, num_labels=91)
#          model weights  → ResNet-50 backbone + transformer encoder/decoder
# model.eval() → disables dropout for deterministic inference
#   training   : dropout in transformer layers regularises attention patterns
#   inference  : must be OFF — you want stable, reproducible attention maps
model = AutoModelForObjectDetection.from_pretrained("facebook/detr-resnet-50")
model.eval()   # IMPORTANT — always call before inference

print("\nSTEP 2 — Model loaded")
print(f"  Model class      : {type(model).__name__}")
print(f"  Num labels       : {model.config.num_labels}")
#   91 — COCO label set (80 thing classes + 11 stuff + background)
print(f"  Num queries      : {model.config.num_queries}")
#   100 — max detectable objects per image
print(f"  Sample labels    : { {k: model.config.id2label[k] for k in range(1, 4)} }")
#   {1: 'N/A', 2: 'bicycle', 3: 'car'}   note: COCO ids are 1-indexed


# ── STEP 3 : Load and preprocess image ───────────────────────────────────────
# Image.open → raw PIL image, mode RGB, size = (width, height)
# processor() applies resize + normalize + to-tensor in one call
# return_tensors="pt" → dict with:
#   pixel_values  : float32 tensor [1, 3, H_resized, W_resized]
#   pixel_mask    : binary tensor  [1, H_resized, W_resized]
#                   1 = real pixel,  0 = padding pixel
#                   DETR uses this so attention ignores padding positions
image  = Image.open("street.png")
inputs = processor(images=image, return_tensors="pt")

print("\nSTEP 3 — Image preprocessed")
print(f"  Original size    : {image.size}")              # (W, H)  e.g. (1280, 720)
print(f"  pixel_values     : {inputs['pixel_values'].shape}")
#   torch.Size([1, 3, 800, 1422])  — height scaled to 800, width proportional
print(f"  pixel_mask       : {inputs['pixel_mask'].shape}")
#   torch.Size([1, 800, 1422])


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() → disables gradient computation
#   inference never calls .backward() — tracking wastes memory for nothing
#   typical saving: ~50% memory, measurable speed gain
#
# Inside DETR — what actually happens:
#   1. Backbone (ResNet-50):
#      image [1, 3, H, W] → feature map [1, 2048, H/32, W/32]
#      e.g. 800×1422 → 25×45 spatial grid
#   2. Projection:
#      Linear(2048 → 256) → [1, 256, 25, 45]
#      flatten spatial → sequence of 25×45 = 1125 tokens, each dim 256
#   3. Encoder (6 layers):
#      self-attention over 1125 feature tokens → global context
#      each token can attend to every other spatial position
#   4. Decoder (6 layers):
#      100 learnable object queries [1, 100, 256]
#      cross-attend to encoder output → each query gathers evidence for one object
#   5. Prediction heads (applied to all 100 query outputs):
#      class head : Linear(256 → 92)     → logits per query  [1, 100, 92]
#      bbox head  : MLP(256 → 256 → 4)  → boxes per query   [1, 100, 4]
#                   box format: (cx, cy, w, h) all normalised to [0, 1]
with torch.no_grad():
    outputs = model(**inputs)

# outputs is a DetrObjectDetectionOutput dataclass
# .logits     → class scores  shape [batch, num_queries, num_labels + 1]
#                              last dim = background/"no object" class
# .pred_boxes → bbox coords   shape [batch, num_queries, 4]
#                              (cx, cy, w, h) normalised — NOT pixel coords yet

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type      : {type(outputs).__name__}")
print(f"  logits shape     : {outputs.logits.shape}")
#   torch.Size([1, 100, 92])
print(f"  pred_boxes shape : {outputs.pred_boxes.shape}")
#   torch.Size([1, 100, 4])


# ── STEP 5 : Post-process bounding boxes ─────────────────────────────────────
# post_process_object_detection does three things:
#   1. Softmax on logits → probabilities across 92 classes per query
#   2. Filter out "no object" class (last index) — keep only real detections
#   3. Rescale boxes from normalised (cx, cy, w, h) → pixel (x_min, y_min, x_max, y_max)
#      using target_sizes as the scale reference
#
# target_sizes must be (height, width) — note the reversal from PIL's (width, height)
# image.size     → (W, H)  e.g. (1280, 720)
# image.size[::-1] → (H, W)  e.g. (720, 1280)  ← what the processor expects
# wrapped in tensor([...]) to match batch dimension
#
# threshold=0.9 → only keep detections where max class probability > 0.9
#   lower threshold → more detections, more false positives
#   higher threshold → fewer detections, higher precision
target_sizes = torch.tensor([image.size[::-1]])   # (height, width)
results = processor.post_process_object_detection(
    outputs,
    target_sizes=target_sizes,
    threshold=0.9
)[0]   # [0] → unwrap batch dimension → dict for the single image

# results dict contains three aligned tensors (one entry per detection):
#   "scores" : float tensor  — confidence probability for the predicted class
#   "labels" : int tensor    — COCO class id
#   "boxes"  : float tensor  — pixel coords [x_min, y_min, x_max, y_max]

print("\nSTEP 5 — Post-processing complete")
print(f"  Detections found : {len(results['scores'])}")


# ── STEP 6 : Print results ────────────────────────────────────────────────────
# zip() iterates all three tensors in lockstep — same index = same detection
# box.tolist()      → tensor [x1, y1, x2, y2] → plain Python list of floats
# round(i, 2)       → trim to 2 decimal places for readability
# label.item()      → tensor scalar → plain Python int → dict key for id2label
# id2label maps COCO integer id → class name string
print("\nSTEP 6 — Detections")
print("─" * 60)
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    box  = [round(i, 2) for i in box.tolist()]
    name = model.config.id2label[label.item()]
    print(f"  {name:<15s} — {score:.3f}   box: {box}")
#   car             — 0.998   box: [10.5, 50.2, 200.3, 150.8]
#   person          — 0.985   box: [220.1, 30.0, 280.5, 200.2]
print("─" * 60)


# ── DETR vs TRADITIONAL DETECTORS — KEY DIFFERENCES ─────────────────────────
# Traditional (Faster R-CNN, YOLO)       DETR
# ──────────────────────────────────────────────────────────────────────────────
# Anchors / region proposals required    No anchors — object queries replace them
# Non-Maximum Suppression (NMS) needed   No NMS — set prediction, no duplicates
# Many hand-designed components          Fully end-to-end trainable
# Fast for small objects                 Struggles with small objects (global attn)
# Conv-only backbone + head              ResNet backbone + full Transformer stack
# ──────────────────────────────────────────────────────────────────────────────


# ── DRY RUN : image = "street.jpg" (1280×720) ────────────────────────────────
#
# Stage 1 — Processor
# 1280×720 → shortest edge (720) scaled to 800 → 1422×800
# pixel_values shape : [1, 3, 800, 1422]
# pixel_mask shape   : [1, 800, 1422]  — all 1s (no padding for single image)
#
# Stage 2 — ResNet-50 backbone
# [1, 3, 800, 1422] → [1, 2048, 25, 45]   (stride 32 downsampling)
# projected → [1, 256, 25, 45]
# flattened → [1, 1125, 256]  (sequence of 1125 spatial tokens)
#
# Stage 3 — Transformer encoder
# 6 layers of self-attention over 1125 tokens
# output : [1, 1125, 256]  — same shape, now context-enriched
#
# Stage 4 — Transformer decoder
# 100 object queries [1, 100, 256] cross-attend to encoder output
# output : [1, 100, 256]  — each query now "describes" one candidate object
#
# Stage 5 — Prediction heads
# class logits  : [1, 100, 92]   — 91 COCO classes + 1 background
# pred_boxes    : [1, 100, 4]    — (cx, cy, w, h) normalised to [0, 1]
#
# Stage 6 — Post-processing (threshold=0.9)
# softmax over 92 → probabilities
# drop background class + apply threshold → e.g. 12 detections survive
# rescale boxes to pixel coords using (720, 1280) target size
# (cx, cy, w, h) → (x_min, y_min, x_max, y_max) in pixels
#
# Final output:
#   car    : 0.998 at [10.5,  50.2,  200.3, 150.8]
#   person : 0.985 at [220.1, 30.0,  280.5, 200.2]

preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

────────────────────────────────────────────────────────────
STEP 1 — Processor loaded
  Processor class  : DetrImageProcessor
  Resize config    : SizeDict(height=None, width=None, longest_edge=1333, shortest_edge=800, max_height=None, max_width=None)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2 — Model loaded
  Model class      : DetrForObjectDetection
  Num labels       : 91
  Num queries      : 100
  Sample labels    : {1: 'person', 2: 'bicycle', 3: 'car'}

STEP 3 — Image preprocessed
  Original size    : (612, 408)
  pixel_values     : torch.Size([1, 3, 800, 1200])
  pixel_mask       : torch.Size([1, 800, 1200])

STEP 4 — Forward pass complete
  Output type      : DetrObjectDetectionOutput
  logits shape     : torch.Size([1, 100, 92])
  pred_boxes shape : torch.Size([1, 100, 4])

STEP 5 — Post-processing complete
  Detections found : 7

STEP 6 — Detections
────────────────────────────────────────────────────────────
  person          — 0.961   box: [296.6, 261.75, 312.31, 287.65]
  car             — 0.994   box: [422.51, 268.65, 451.55, 283.62]
  bicycle         — 0.957   box: [199.76, 266.47, 225.07, 291.66]
  bicycle         — 0.906   box: [0.6, 275.37, 43.73, 300.6]
  person          — 0.961   box: [315.39, 265.09, 323.71, 286.42]
  bicycle         — 0.999   box

In [9]:
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation
from PIL import Image
import torch
import numpy as np

# ── AUTOMODEL FOR SEMANTIC SEGMENTATION (nvidia/segformer-b5-finetuned-ade-640-640) ──
# Task         : Semantic segmentation — assign a class label to every single pixel
#                unlike object detection (bounding boxes) or instance seg (per-object masks)
#                every pixel gets exactly one class — "sky", "wall", "road", "person" etc.
# Architecture : SegFormer-B5 — hierarchical Vision Transformer encoder + lightweight MLP decoder
#                Encoder : Mix Transformer (MiT-B5) — 4 stages, each at lower resolution
#                          stage 1: H/4  × W/4   — fine local features
#                          stage 2: H/8  × W/8
#                          stage 3: H/16 × W/16
#                          stage 4: H/32 × W/32  — coarse semantic features
#                Decoder : All4One MLP — fuses all 4 stage outputs → [B, 768, H/4, W/4]
#                          final Linear(768 → num_labels) → logits [B, num_labels, H/4, W/4]
# Fine-tuned on: ADE20K — 150 semantic categories (indoor + outdoor scenes)
# Key insight  : No positional encoding → resolution-agnostic at inference time
#                logit map is H/4 × W/4 — must be upsampled back to original resolution
# Output       : low-res logits → bilinear upsample → argmax per pixel → segmentation map
# ─────────────────────────────────────────────────────────────────────────────────────────────


# ── STEP 1 : Load processor ───────────────────────────────────────────────────
# AutoImageProcessor for SegFormer wraps:
#   resize      → 640×640 (model's training resolution — both H and W fixed here)
#   normalize   → ImageNet mean/std per channel
#                 mean = [0.485, 0.456, 0.406]  std = [0.229, 0.224, 0.225]
#   to tensor   → PIL HWC uint8 → PyTorch float CHW tensor
# Note: ADE20K scenes are complex — 640×640 preserves fine detail vs smaller inputs
processor = AutoImageProcessor.from_pretrained(
    "nvidia/segformer-b5-finetuned-ade-640-640"
)

print("─" * 60)
print("STEP 1 — Processor loaded")
print(f"  Processor class  : {type(processor).__name__}")
print(f"  Image size       : {processor.size}")
#   {'height': 640, 'width': 640}  — fixed square, both dims specified


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForSemanticSegmentation adds the All4One MLP decoder head on top of MiT-B5
# Downloads config.json → architecture details (num_labels=150, id2label mapping)
#          model weights → MiT-B5 encoder + MLP decoder
# model.eval() → disables dropout for deterministic inference
#   training   : dropout + stochastic depth in transformer layers → regularisation
#   inference  : must be OFF — full stable feature maps required
model = AutoModelForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b5-finetuned-ade-640-640"
)
model.eval()   # IMPORTANT — always call before inference

print("\nSTEP 2 — Model loaded")
print(f"  Model class      : {type(model).__name__}")
print(f"  Num labels       : {model.config.num_labels}")
#   150 — full ADE20K semantic category set
print(f"  Sample labels    : { {k: model.config.id2label[k] for k in range(4)} }")
#   {0: 'wall', 1: 'building', 2: 'sky', 3: 'floor'}


# ── STEP 3 : Load and preprocess image ───────────────────────────────────────
# Image.open → raw PIL image, mode RGB, size = (width, height)
# processor() resizes to 640×640, normalises, converts to tensor
# return_tensors="pt" → output dict with:
#   pixel_values : float32 tensor [1, 3, 640, 640]
# Note: resizing to a fixed square distorts aspect ratio —
#       original proportions are NOT preserved here (unlike DETR)
#       the upsample in Step 5 restores predictions to original dimensions
image  = Image.open("images/scene.png")
inputs = processor(images=image, return_tensors="pt")

print("\nSTEP 3 — Image preprocessed")
print(f"  Original size    : {image.size}")
#   (W, H) e.g. (1024, 768)
print(f"  pixel_values     : {inputs['pixel_values'].shape}")
#   torch.Size([1, 3, 640, 640])


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() → disables gradient tracking — never needed at inference
#   saves ~50% memory and speeds up forward pass meaningfully
#
# Inside SegFormer — what actually happens:
#   MiT-B5 encoder — 4 hierarchical stages:
#     stage 1 : overlapping patch embed 4×4 stride 4 → [1, 64,  160, 160]
#               2 transformer blocks with Efficient Self-Attention (ESA)
#               ESA reduces key/value sequence by factor R → cheaper attention
#     stage 2 : patch embed 3×3 stride 2 → [1, 128,  80,  80]
#               2 transformer blocks
#     stage 3 : patch embed 3×3 stride 2 → [1, 320,  40,  40]
#               2 transformer blocks
#     stage 4 : patch embed 3×3 stride 2 → [1, 512,  20,  20]
#               2 transformer blocks
#   MLP decoder — All4One fusion:
#     each stage output passed through Linear → unified channel dim 768
#     all 4 outputs upsampled to stage-1 resolution (H/4 = 160)
#     concatenated → [1, 768×4, 160, 160]  then Linear(768×4 → 768)
#     final Linear(768 → 150) → logits [1, 150, 160, 160]
with torch.no_grad():
    outputs = model(**inputs)

# outputs is a SemanticSegmenterOutput dataclass
# .logits → raw class scores per spatial position
#            shape [batch_size, num_labels, height/4, width/4]
#            NOT full resolution — only 1/4 the input spatial size
#            each of 150 channels = score for that class at that position
logits = outputs.logits   # shape: [1, 150, 160, 160]

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type      : {type(outputs).__name__}")
print(f"  Logits shape     : {logits.shape}")
#   torch.Size([1, 150, 160, 160])
#   note: 640/4 = 160 — output is quarter resolution of input


# ── STEP 5 : Upsample logits to original image size ──────────────────────────
# Model outputs at 1/4 input resolution → must upsample back to original image size
# interpolate() resizes the spatial dims of a 4-D tensor [B, C, H, W]
#   input  : [1, 150, 160, 160]
#   output : [1, 150, H_orig, W_orig]
#
# size=image.size[::-1]  — why the reversal?
#   image.size     → PIL reports (width, height)
#   interpolate()  → expects (height, width)
#   [::-1]         → flips the tuple  e.g. (1024, 768) → (768, 1024)
#
# mode="bilinear"  → smooth interpolation across H and W
#   nearest        → blocky — each new pixel copies its nearest original pixel
#   bilinear       → weighted average of 4 neighbours — smoother boundaries
# align_corners=False → standard for dense prediction — corners treated as pixel edges
#                        use False to match how the processor downsampled the image
upsampled = torch.nn.functional.interpolate(
    logits,
    size=image.size[::-1],   # (height, width) of original image
    mode="bilinear",
    align_corners=False
)

print("\nSTEP 5 — Logits upsampled")
print(f"  Upsampled shape  : {upsampled.shape}")
#   torch.Size([1, 150, 768, 1024])  — now matches original image resolution


# ── STEP 6 : Argmax — class prediction per pixel ─────────────────────────────
# argmax(dim=1) → across the 150-class channel dimension
#   for each pixel position (h, w) → find which of 150 classes has the highest logit
#   output shape: [1, H, W]  — one integer class id per pixel
# [0] → remove batch dimension → final shape [H, W]
# result: a 2-D array where each value is a class id in range [0, 149]
pred_seg = upsampled.argmax(dim=1)[0]   # shape: (H, W)

print("\nSTEP 6 — Pixel-wise predictions")
print(f"  pred_seg shape   : {pred_seg.shape}")
#   torch.Size([768, 1024])
print(f"  Unique classes   : {pred_seg.unique().tolist()}")
#   [0, 2, 3, 21, 53, ...]  — only classes actually present in the image
print(f"  Class breakdown  :")
for class_id in pred_seg.unique().tolist():
    pixel_count = (pred_seg == class_id).sum().item()
    pct = pixel_count / pred_seg.numel() * 100
    print(f"    {model.config.id2label[class_id]:<20s} — {pct:.1f}%")
#   wall                 — 31.2%
#   sky                  — 22.8%
#   floor                — 18.5%
#   building             — 14.1%
#   tree                 — 8.3%


# ── STEP 7 : Colorize segmentation map ───────────────────────────────────────
# model.config.id2label is a dict {int: str} — NOT an array of colors
# for a visual colorized map you need a (num_labels, 3) RGB palette array
# ADE20K has a standard 150-class color palette — build it explicitly
#
# Simple approach — generate a deterministic color per class id:
#   multiply class_id by a prime-spaced offset across RGB channels
#   guarantees visually distinct colors for nearby class ids
palette = np.zeros((model.config.num_labels, 3), dtype=np.uint8)
for class_id in range(model.config.num_labels):
    palette[class_id] = [
        (class_id * 73) % 256,    # R channel
        (class_id * 137) % 256,   # G channel
        (class_id * 211) % 256,   # B channel
    ]

# map each pixel's class id → its RGB color
# pred_seg.numpy() → [H, W] int array of class ids
# palette[pred_seg_np] → [H, W, 3] RGB array via NumPy fancy indexing
pred_seg_np  = pred_seg.numpy().astype(np.uint8)   # [H, W]
colored      = palette[pred_seg_np]                # [H, W, 3]  — RGB color per pixel

# Image.fromarray → convert [H, W, 3] uint8 numpy array → PIL RGB image
seg_img = Image.fromarray(colored)

print("\nSTEP 7 — Segmentation map colorized")
print(f"  seg_img size     : {seg_img.size}")
#   (1024, 768)  — matches original image resolution
print(f"  seg_img mode     : {seg_img.mode}")
#   RGB

seg_img.save("segmentation_output.png")
print("  Saved            : segmentation_output.png")


# ── SEGMENTATION TYPES — KEY DIFFERENCES ─────────────────────────────────────
# Type                   Output                  Distinguishes instances?
# ──────────────────────────────────────────────────────────────────────────────
# Semantic segmentation  Class label per pixel   No — all cars same color
# Instance segmentation  Mask per object         Yes — car1 / car2 separate
# Panoptic segmentation  Semantic + instance     Yes — things + stuff unified
# ──────────────────────────────────────────────────────────────────────────────
# SegFormer does semantic only — two adjacent cars get identical label/color


# ── DRY RUN : image = "scene.jpg" (1024×768) ─────────────────────────────────
#
# Stage 1 — Processor
# 1024×768 → resize → 640×640  (aspect ratio distorted — both dims fixed)
# normalize per channel with ImageNet stats
# pixel_values : [1, 3, 640, 640]
#
# Stage 2 — MiT-B5 encoder
# stage 1 : patch 4×4 stride 4 → [1, 64,  160, 160]  — 2 transformer blocks
# stage 2 : patch 3×3 stride 2 → [1, 128,  80,  80]  — 2 transformer blocks
# stage 3 : patch 3×3 stride 2 → [1, 320,  40,  40]  — 2 transformer blocks
# stage 4 : patch 3×3 stride 2 → [1, 512,  20,  20]  — 2 transformer blocks
#
# Stage 3 — All4One MLP decoder
# each stage → Linear → 768 channels → upsample to 160×160
# concat all 4 → [1, 3072, 160, 160] → Linear(3072 → 768)
# Linear(768 → 150) → logits [1, 150, 160, 160]
#
# Stage 4 — Upsample
# [1, 150, 160, 160] → bilinear → [1, 150, 768, 1024]
#
# Stage 5 — Argmax
# [1, 150, 768, 1024] → argmax dim=1 → [768, 1024]
# each pixel = one integer in [0, 149]
#
# Stage 6 — Colorize
# palette[pred_seg_np] → [768, 1024, 3]  → PIL RGB image
# sky pixels → all same color, wall pixels → all same color etc.

────────────────────────────────────────────────────────────
STEP 1 — Processor loaded
  Processor class  : SegformerImageProcessor
  Image size       : SizeDict(height=640, width=640, longest_edge=None, shortest_edge=None, max_height=None, max_width=None)


Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]


STEP 2 — Model loaded
  Model class      : SegformerForSemanticSegmentation
  Num labels       : 150
  Sample labels    : {0: 'wall', 1: 'building', 2: 'sky', 3: 'floor'}

STEP 3 — Image preprocessed
  Original size    : (612, 408)
  pixel_values     : torch.Size([1, 3, 640, 640])

STEP 4 — Forward pass complete
  Output type      : SemanticSegmenterOutput
  Logits shape     : torch.Size([1, 150, 160, 160])

STEP 5 — Logits upsampled
  Upsampled shape  : torch.Size([1, 150, 408, 612])

STEP 6 — Pixel-wise predictions
  pred_seg shape   : torch.Size([408, 612])
  Unique classes   : [0, 1, 6, 11, 12, 115]
  Class breakdown  :
    wall                 — 0.5%
    building             — 55.6%
    road                 — 0.2%
    sidewalk             — 4.3%
    person               — 39.3%
    bag                  — 0.1%

STEP 7 — Segmentation map colorized
  seg_img size     : (612, 408)
  seg_img mode     : RGB
  Saved            : segmentation_output.png


In [10]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from PIL import Image
import torch

# ── AUTOMODEL FOR DEPTH ESTIMATION (Intel/dpt-large) ─────────────────────────
# Task         : Monocular depth estimation — predict a depth value for every pixel
#                from a single RGB image (no stereo camera, no LiDAR)
#                output is a relative depth map — closer objects → lower values
#                                                 farther objects → higher values
# Architecture : DPT-Large (Dense Prediction Transformer)
#                Backbone  : ViT-Large — 24 transformer layers, hidden dim 1024
#                            image split into 16×16 patches → sequence of tokens
#                Neck      : Reassemble + Fusion blocks
#                            taps into 4 intermediate ViT layers (not just the last)
#                            progressively reassembles patch tokens → spatial feature maps
#                            at 4 resolutions: /32, /16, /8, /4 of input
#                            RefineNet-style fusion merges all 4 → rich dense features
#                Head      : Convolutional head → single-channel depth map at H/2 × W/2
#                            values are relative inverse depth (disparity-like)
# Pre-trained  : Mix 6 — trained on 1.4M+ images across 6 diverse depth datasets
# Key insight  : ViT's global self-attention captures long-range depth cues
#                (e.g. vanishing point, sky vs floor) better than pure CNNs
# Output       : low-res depth map → bicubic upsample → normalise → visualise
# ─────────────────────────────────────────────────────────────────────────────


# ── STEP 1 : Load processor ───────────────────────────────────────────────────
# AutoImageProcessor for DPT wraps:
#   resize      → 384×384  (ViT-Large's fixed input resolution)
#                 both dims fixed — aspect ratio NOT preserved
#   normalize   → ImageNet mean/std per channel
#                 mean = [0.485, 0.456, 0.406]  std = [0.229, 0.224, 0.225]
#   to tensor   → PIL HWC uint8 → PyTorch float CHW tensor
# Must match the exact preprocessing used during pre-training —
# wrong image size or normalisation silently degrades depth predictions
processor = AutoImageProcessor.from_pretrained("Intel/dpt-large")

print("─" * 60)
print("STEP 1 — Processor loaded")
print(f"  Processor class  : {type(processor).__name__}")
print(f"  Image size       : {processor.size}")
#   {'height': 384, 'width': 384}


# ── STEP 2 : Load model ───────────────────────────────────────────────────────
# AutoModelForDepthEstimation loads DPT with the depth head
# Downloads config.json → architecture details (hidden_size=1024, num_layers=24)
#          model weights → ViT-Large backbone + Reassemble/Fusion neck + depth head
# model.eval() → disables dropout for deterministic inference
#   training   : dropout in transformer layers → regularisation
#   inference  : must be OFF — deterministic depth values required
model = AutoModelForDepthEstimation.from_pretrained("Intel/dpt-large")
model.eval()   # IMPORTANT — always call before inference

print("\nSTEP 2 — Model loaded")
print(f"  Model class      : {type(model).__name__}")
print(f"  Hidden size      : {model.config.hidden_size}")
#   1024  — ViT-Large (vs 768 for ViT-Base used in previous examples)
print(f"  Num layers       : {model.config.num_hidden_layers}")
#   24    — ViT-Large (vs 12 for ViT-Base)


# ── STEP 3 : Load and preprocess image ───────────────────────────────────────
# Image.open → raw PIL image, mode RGB, size = (width, height)
# Store original size now — needed later to upsample depth back to full resolution
# processor() resizes to 384×384, normalises, converts to tensor
# return_tensors="pt" → dict with:
#   pixel_values : float32 tensor [1, 3, 384, 384]
image = Image.open("images/room.png")
inputs = processor(images=image, return_tensors="pt")

print("\nSTEP 3 — Image preprocessed")
print(f"  Original size    : {image.size}")
#   (W, H) e.g. (1280, 960)
print(f"  pixel_values     : {inputs['pixel_values'].shape}")
#   torch.Size([1, 3, 384, 384])


# ── STEP 4 : Forward pass ─────────────────────────────────────────────────────
# torch.no_grad() → disables gradient tracking — never needed at inference
#   saves ~50% memory and speeds up forward pass
#
# Inside DPT-Large — what actually happens:
#   1. Patch embed (ViT):
#      384×384 image → 24×24 grid of 16×16 patches → 576 patch tokens
#      prepend [CLS] → sequence length 577, each token dim 1024
#   2. ViT-Large transformer (24 layers):
#      full self-attention over all 577 tokens
#      DPT taps intermediate outputs at layers 6, 12, 18, 24
#      → 4 feature tensors, each [1, 577, 1024]
#   3. Reassemble blocks (one per tapped layer):
#      drop [CLS] token → reshape 576 tokens → spatial grids
#      layer 6  → [1, 256, 48, 48]   (H/8  of 384)
#      layer 12 → [1, 512, 24, 24]   (H/16 of 384)
#      layer 18 → [1, 512, 12, 12]   (H/32 of 384)
#      layer 24 → [1, 512,  6,  6]   (H/64 of 384)
#   4. Fusion blocks (RefineNet-style):
#      progressively upsample + fuse from coarsest to finest
#      output : [1, 256, 192, 192]  (H/2 of 384)
#   5. Depth head:
#      Conv → ReLU → Conv(256 → 1) → [1, 1, 192, 192]
#      squeeze → [1, 192, 192]  — single-channel relative depth map
with torch.no_grad():
    outputs = model(**inputs)

# outputs is a DepthEstimatorOutput dataclass
# .predicted_depth → relative depth values, shape [batch_size, height, width]
#   NO channel dimension — depth is inherently single-channel
#   values are NOT metric distances (not in metres) — relative/inverse depth
#   higher value = farther from camera (for DPT — some models invert this)
depth = outputs.predicted_depth   # shape: [1, 192, 192]

print("\nSTEP 4 — Forward pass complete")
print(f"  Output type      : {type(outputs).__name__}")
print(f"  predicted_depth  : {depth.shape}")
#   torch.Size([1, 192, 192])  — half of 384 input resolution
print(f"  Depth range      : [{depth.min().item():.3f}, {depth.max().item():.3f}]")
#   [1.243, 87.651]  — relative units, not metres


# ── STEP 5 : Upsample depth map to original image size ───────────────────────
# interpolate() requires 4-D input [B, C, H, W] — depth is 3-D [B, H, W]
# .unsqueeze(1) → inserts channel dim at position 1 → [1, 1, 192, 192]
#
# size=image.size[::-1] — why the reversal?
#   image.size     → PIL reports (width, height)  e.g. (1280, 960)
#   interpolate()  → expects (height, width)       e.g. (960, 1280)
#   [::-1]         → flips the tuple
#
# mode="bicubic"  → fits a cubic polynomial across 4×4 neighbourhood
#   nearest        → blocky, sharp edges — bad for smooth depth gradients
#   bilinear       → linear between 4 neighbours — smooth but slightly blurry
#   bicubic        → smoother curves, better for continuous depth surfaces
#                    preferred for depth maps — surfaces are rarely flat planes
# align_corners=False → standard convention for dense prediction tasks
#
# .squeeze() → removes both batch (dim 0) and channel (dim 1) → [H_orig, W_orig]
depth_upsampled = torch.nn.functional.interpolate(
    depth.unsqueeze(1),          # [1, 1, 192, 192]
    size=image.size[::-1],       # (height, width) of original image
    mode="bicubic",
    align_corners=False
).squeeze()                      # [H_orig, W_orig]

print("\nSTEP 5 — Depth map upsampled")
print(f"  Upsampled shape  : {depth_upsampled.shape}")
#   torch.Size([960, 1280])  — matches original image resolution


# ── STEP 6 : Convert to numpy ─────────────────────────────────────────────────
# .numpy() → PyTorch tensor → NumPy float32 array
# required because PIL.Image.fromarray() only accepts NumPy arrays
# shape stays (H, W) — single-channel, no change needed
depth_numpy = depth_upsampled.numpy()   # shape: (960, 1280)  dtype: float32

print("\nSTEP 6 — Converted to NumPy")
print(f"  dtype            : {depth_numpy.dtype}")
#   float32
print(f"  shape            : {depth_numpy.shape}")
#   (960, 1280)


# ── STEP 7 : Normalise for visualisation ──────────────────────────────────────
# Raw depth values are relative — range depends on scene content entirely
# To display as a grayscale image, must map to [0, 255] uint8
#
# Min-max normalisation:
#   depth_normalized = (x - min) / (max - min)
#   guarantees output range exactly [0.0, 1.0]
#   closest object → 0.0 (black)   farthest object → 1.0 (white)
#
# * 255   → scale [0.0, 1.0] → [0.0, 255.0]
# .astype("uint8") → truncate to integer [0, 255]
#
# Limitation: min-max normalisation is scene-relative — absolute distances
# are not preserved. Two images of different scenes are not comparable.
# For metric depth, use a metric model (e.g. ZoeDepth) instead.
depth_normalized = (depth_numpy - depth_numpy.min()) / (
    depth_numpy.max() - depth_numpy.min()
)

depth_img = Image.fromarray((depth_normalized * 255).astype("uint8"))

print("\nSTEP 7 — Normalised and converted to image")
print(f"  Normalised range : [{depth_normalized.min():.4f}, {depth_normalized.max():.4f}]")
#   [0.0000, 1.0000]  — always exactly this after min-max
print(f"  depth_img size   : {depth_img.size}")
#   (1280, 960)  — matches original image
print(f"  depth_img mode   : {depth_img.mode}")
#   L  — 8-bit greyscale

depth_img.save("depth_output.png")
print("  Saved            : depth_output.png")


# ── DEPTH ESTIMATION TYPES — KEY DIFFERENCES ─────────────────────────────────
# Type                  Input           Output           Metric?
# ──────────────────────────────────────────────────────────────────────────────
# Monocular relative    Single RGB      Relative depth   No  — scene-relative only
# Monocular metric      Single RGB      Absolute depth   Yes — values in metres
# Stereo                RGB pair        Disparity map    Yes — with calibration
# RGB-D                 RGB + depth     Fused map        Yes — sensor ground truth
# ──────────────────────────────────────────────────────────────────────────────
# DPT-Large = monocular relative — great for visualisation, not for robotics


# ── VISUALISATION OPTIONS ─────────────────────────────────────────────────────
# Greyscale (above)  : simple, one channel, no extra dependencies
# Colormap           : apply matplotlib / OpenCV colourmap for perceptual clarity
#                      closer = warm (red), farther = cool (blue) — more readable
#
# import matplotlib.pyplot as plt
# plt.imshow(depth_normalized, cmap="plasma")  # or "inferno", "magma", "viridis"
# plt.colorbar(label="Relative depth")
# plt.axis("off")
# plt.savefig("depth_colormap.png", bbox_inches="tight")


# ── DRY RUN : image = "room.jpg" (1280×960) ──────────────────────────────────
#
# Stage 1 — Processor
# 1280×960 → resize → 384×384  (aspect ratio distorted)
# normalize with ImageNet stats
# pixel_values : [1, 3, 384, 384]
#
# Stage 2 — ViT-Large backbone
# 384×384 → 576 patches of 16×16 → sequence [1, 577, 1024]  (577 = 576 + CLS)
# 24 transformer layers — intermediate outputs tapped at layers 6, 12, 18, 24
#
# Stage 3 — Reassemble blocks
# layer  6 output → spatial [1, 256, 48, 48]
# layer 12 output → spatial [1, 512, 24, 24]
# layer 18 output → spatial [1, 512, 12, 12]
# layer 24 output → spatial [1, 512,  6,  6]
#
# Stage 4 — Fusion blocks (coarse → fine)
# [1, 512, 6, 6] → upsample + fuse → ... → [1, 256, 192, 192]
#
# Stage 5 — Depth head
# Conv(256 → 1) → [1, 1, 192, 192] → squeeze → [1, 192, 192]
# predicted_depth range e.g. [1.24, 87.65]
#
# Stage 6 — Upsample
# [1, 1, 192, 192] → bicubic → [1, 1, 960, 1280] → squeeze → [960, 1280]
#
# Stage 7 — Normalise
# (x - 1.24) / (87.65 - 1.24) → [0.0, 1.0] → * 255 → uint8
# PIL Image (960×1280) mode L — greyscale depth visualisation

preprocessor_config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

────────────────────────────────────────────────────────────
STEP 1 — Processor loaded
  Processor class  : DPTImageProcessor
  Image size       : SizeDict(height=384, width=384, longest_edge=None, shortest_edge=None, max_height=None, max_width=None)


config.json:   0%|          | 0.00/942 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.37G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

DPTForDepthEstimation LOAD REPORT from: Intel/dpt-large
Key                                                            | Status  | 
---------------------------------------------------------------+---------+-
neck.fusion_stage.layers.0.residual_layer1.convolution2.weight | MISSING | 
neck.fusion_stage.layers.0.residual_layer1.convolution1.weight | MISSING | 
neck.fusion_stage.layers.0.residual_layer1.convolution1.bias   | MISSING | 
neck.fusion_stage.layers.0.residual_layer1.convolution2.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



STEP 2 — Model loaded
  Model class      : DPTForDepthEstimation
  Hidden size      : 1024
  Num layers       : 24

STEP 3 — Image preprocessed
  Original size    : (800, 534)
  pixel_values     : torch.Size([1, 3, 384, 384])

STEP 4 — Forward pass complete
  Output type      : DepthEstimatorOutput
  predicted_depth  : torch.Size([1, 384, 384])
  Depth range      : [0.000, 23.950]

STEP 5 — Depth map upsampled
  Upsampled shape  : torch.Size([534, 800])

STEP 6 — Converted to NumPy
  dtype            : float32
  shape            : (534, 800)

STEP 7 — Normalised and converted to image
  Normalised range : [0.0000, 1.0000]
  depth_img size   : (800, 534)
  depth_img mode   : L
  Saved            : depth_output.png
